# Evidence viewer (optional)

For one volume of a trained run: **which slices** drove each predicted label (exact per-slice
contributions to the logit) and, on request, **where inside those slices** the model looked
(Grad-CAM on the encoder's patch tokens). Same functions as `scripts/model/explain_volume.py`.

Needs the HU cache and the embedding store, so run it on the server. Heatmaps re-encode a few slices
with DALE-CT-2S: use a GPU.

**Read with care:** this shows what the *model* relied on. It is not a validated localisation of the
finding, and slice numbers are slices of the *preprocessed* volume (1.5 mm, head to foot, body-cropped).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# run from the repository root, wherever the notebook was opened
root = Path.cwd().resolve()
while not (root / "configs" / "preprocessing.yaml").exists() and root != root.parent:
    root = root.parent
os.chdir(root)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 1. Choose the run and the volume

In [ ]:
from ct_model.explain.volume import open_explainer, plot_label

RUN_DIR = None        # outputs/experiments/<name>/<config hash>/seed<k>; None = the most recently finished run
VOLUME_ID = None      # None = the first test volume
LABELS = None         # None = the labels predicted positive at the val thresholds
TOP_K = 5
HEATMAPS = False      # True: in-slice maps for the top slices (re-encodes them)
DEVICE = "auto"

if RUN_DIR is None:
    ledger = pd.read_csv("outputs/experiments/index.csv")
    RUN_DIR = ledger.sort_values("finished").dir.iloc[-1]
ex = open_explainer(RUN_DIR, device=DEVICE)
vid = VOLUME_ID or ex.volume_ids("test")[0]
ev = ex.evidence(vid)
truth = ex.labels(vid)
rec = ex.record(vid)
print(f"{vid}: split {rec.split}, kernel {rec.kernel_class}, {rec.n_slices} slices")
table = pd.DataFrame({"probability": ev.probs, "val_threshold": ev.thresholds, "predicted": ev.predicted,
                      "manifest_label": truth}, index=ev.label_names).sort_values("probability", ascending=False)
display(table.round(3))

## 2. Evidence per label

Top: every slice's contribution to the label's logit (red pushes towards "present", blue away).
Bottom: the top slices in a lung window, with the heatmap overlay if `HEATMAPS = True`.

In [ ]:
labels = LABELS or [n for n, p in zip(ev.label_names, ev.predicted) if p]
if not labels:
    print("no label predicted positive -- set LABELS to look at specific ones")
hu = ex.hu(vid)
for label in labels:
    c = ev.label_names.index(label)
    maps = ex.heatmaps(vid, label, ev.top_slices(c, TOP_K)) if HEATMAPS else None
    plot_label(ev, label, hu, maps, k=TOP_K, true_value=truth[c])
    plt.show()
    if maps:
        print("re-encode cosine (stored vs recomputed embedding):", [round(m.reencode_cosine, 4) for m in maps])

## 3. Sanity checks for the heatmaps (run before trusting them)

* **Randomisation** (Adebayo et al. 2018): with the head's weights randomised the map must change. A
  correlation near 1 means the map does not depend on what the model learned.
* **Deletion**: blanking the top-ranked patches should lower the logit more than blanking random ones.

In [ ]:
RUN_CHECKS = False
if RUN_CHECKS and labels:
    from ct_model.explain.heatmap import deletion_check, randomization_check, slice_heatmap

    label = labels[0]
    k = int(ev.top_slices(label, 1)[0])
    bag, s = ex.bag(vid), np.array(hu[k])
    r = randomization_check(ex.trained.model, ex.encoder(), bag, s, k, label, ex.device)
    h = slice_heatmap(ex.trained.model, ex.encoder(), bag, s, k, label, ex.device)
    d = deletion_check(ex.trained.model, ex.encoder(), bag, s, k, label, h.patch_cam, device=ex.device)
    print(f"{label}, slice {k}: randomisation correlation {r:.3f} (should be well below 1)")
    print(f"deletion: logit drop {d['drop_top']:.3f} for the top {d['fraction']:.0%} patches vs {d['drop_random_mean']:.3f} for random ones")